In [1]:
# CELL 0 – dùng chung cho NB01 → NB07
import os, json
from pathlib import Path

PROJECT_ROOT = Path(os.getcwd()).parent
CONFIG_PATH = PROJECT_ROOT / "src" / "config.json"

if not CONFIG_PATH.exists():
    raise FileNotFoundError(f"Không tìm thấy {CONFIG_PATH}. Hãy chạy 00_prep_features.ipynb trước.")

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    cfg = json.load(f)

RAW      = Path(cfg["RAW"])       
FEATURES = Path(cfg["FEATURES"])
RESULTS  = Path(cfg["RESULTS"])
FIGURES  = Path(cfg["FIGURES"])

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RAW     :", RAW)
print("FEATURES:", FEATURES)
print("RESULTS :", RESULTS)
print("FIGURES :", FIGURES)


PROJECT_ROOT: D:\STAT3013.Q12_Group01
RAW     : D:\STAT3013.Q12_Group01\data\raw
FEATURES: D:\STAT3013.Q12_Group01\features
RESULTS : D:\STAT3013.Q12_Group01\results
FIGURES : D:\STAT3013.Q12_Group01\figures


In [2]:
import pandas as pd
import numpy as np

tx_raw      = pd.read_csv(RAW / "transaction_data.csv")
product_raw = pd.read_csv(RAW / "product.csv")
causal_raw  = pd.read_csv(RAW / "causal_data.csv")

print(tx_raw.shape, product_raw.shape, causal_raw.shape)
tx_raw.head()


(1048575, 12) (92353, 7) (1048575, 5)


,household_key,BASKET_ID,DAY,PRODUCT_ID,QUANTITY,SALES_VALUE,STORE_ID,RETAIL_DISC,TRANS_TIME,WEEK_NO,COUPON_DISC,COUPON_MATCH_DISC
0,2375,26984851472,1,1004906,1,1.39,364,-0.60,1631,1,0.0,0.0
1,2375,26984851472,1,1033142,1,0.82,364,0.00,1631,1,0.0,0.0
2,2375,26984851472,1,1036325,1,0.99,364,-0.30,1631,1,0.0,0.0
3,2375,26984851472,1,1082185,1,1.21,364,0.00,1631,1,0.0,0.0
4,2375,26984851472,1,8160430,1,1.50,364,-0.39,1631,1,0.0,0.0


In [3]:
def lower_cols(df):
    df = df.copy()
    df.columns = df.columns.str.lower()
    return df

tx = lower_cols(tx_raw)
product = lower_cols(product_raw)
causal_data = lower_cols(causal_raw)

tx.columns.tolist()


['household_key',
 'basket_id',
 'day',
 'product_id',
 'quantity',
 'sales_value',
 'store_id',
 'retail_disc',
 'trans_time',
 'week_no',
 'coupon_disc',
 'coupon_match_disc']

In [4]:
tx = tx[(tx.quantity > 0) & (tx.sales_value >= 0)].copy()

tx["unit_price"] = tx["sales_value"] / tx["quantity"]
tx["unit_price"] = tx["unit_price"].replace([np.inf, -np.inf], np.nan)

lo, hi = tx["unit_price"].quantile([0.005, 0.995])
tx["unit_price"] = tx["unit_price"].clip(lo, hi)



In [5]:
tx = tx.merge(
    product[["product_id","department","commodity_desc","sub_commodity_desc"]],
    on="product_id", how="left"
)



In [6]:
promo_cols = ["store_id","week_no","product_id","display","mailer","price_reduction"]
promo_cols = [c for c in promo_cols if c in causal_data.columns]

promo = causal_data[promo_cols].copy()
promo["week_no"] = promo["week_no"].astype(int)
tx["week_no"] = tx["week_no"].astype(int)

tx = tx.merge(promo, on=["store_id","week_no","product_id"], how="left")
for c in ["display","mailer","price_reduction"]:
    if c in tx.columns:
        tx[c] = tx[c].fillna(0)

tx.head()


C:\Users\CHI LAM\AppData\Local\Temp\ipykernel_30908\2408802077.py:11: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  tx[c] = tx[c].fillna(0)


,household_key,basket_id,day,product_id,quantity,sales_value,store_id,retail_disc,trans_time,week_no,coupon_disc,coupon_match_disc,unit_price,department,commodity_desc,sub_commodity_desc,display,mailer
0,2375,26984851472,1,1004906,1,1.39,364,-0.60,1631,1,0.0,0.0,1.39,PRODUCE,POTATOES,POTATOES RUSSET (BULK&BAG),0,0
1,2375,26984851472,1,1033142,1,0.82,364,0.00,1631,1,0.0,0.0,0.82,PRODUCE,ONIONS,ONIONS SWEET (BULK&BAG),0,0
2,2375,26984851472,1,1036325,1,0.99,364,-0.30,1631,1,0.0,0.0,0.99,PRODUCE,VEGETABLES - ALL OTHERS,CELERY,0,0
3,2375,26984851472,1,1082185,1,1.21,364,0.00,1631,1,0.0,0.0,1.21,PRODUCE,TROPICAL FRUIT,BANANAS,0,0
4,2375,26984851472,1,8160430,1,1.50,364,-0.39,1631,1,0.0,0.0,1.50,PRODUCE,ORGANICS FRUIT & VEGETABLES,ORGANIC CARROTS,0,0


In [7]:
# Descriptive Statistics cho các biến chính
import pandas as pd

desc_cols = ["sales_value", "quantity", "unit_price"]
desc_table = tx[desc_cols].describe().T

RESULTS.mkdir(exist_ok=True)
out_path = RESULTS / "nb01_descriptive_summary.csv"
desc_table.to_csv(out_path)
print("Saved descriptive summary to:", out_path)

desc_table


Saved descriptive summary to: D:\STAT3013.Q12_Group01\results\nb01_descriptive_summary.csv


,count,mean,std,min,25%,50%,75%,max
sales_value,1043543.0,3.066138,4.016015,0.000000,1.29,2.00,3.49,505.00
quantity,1043543.0,94.614526,1114.719218,1.000000,1.00,1.00,1.00,85055.00
unit_price,1043543.0,2.370400,2.117394,0.002139,1.00,1.89,2.99,15.23


In [8]:
# Save cleaned transaction data
FEATURES.mkdir(exist_ok=True)
out_path = FEATURES / "tx_clean.parquet"
tx.to_parquet(out_path, index=False)
print("Saved tx_clean.parquet to:", out_path)


Saved tx_clean.parquet to: D:\STAT3013.Q12_Group01\features\tx_clean.parquet
